# Notebook 10 — QCD Running Coupling, Clockwork Chain Lengths, and Schwarzschild Constraint Violation

**Companion to:** `fundamental_forces_planck_scale.md`

Four computational sections:
1. QCD running coupling vs Stribeck curve fit
2. Clockwork chain lengths for multiple thresholds
3. Quadratic maximum verification (MOND and QCD beta function)
4. Schwarzschild mimetic constraint violation

Uses only Python standard library. CC0.

**References:**
- Gross, D. J. & Wilczek, F. (1973). *Phys. Rev. Lett.*, 30, 1343.
- Politzer, H. D. (1973). *Phys. Rev. Lett.*, 30, 1346.
- Giudice, G. F. & McCullough, M. (2017). *JHEP*, 02, 036.
- Feigenbaum, M. J. (1978). *J. Stat. Phys.*, 19, 25.
- Chamseddine, A. H. & Mukhanov, V. (2013). *JHEP*, 11, 135.
- Penrose, R. (1965). *Phys. Rev. Lett.*, 14, 57.

In [ ]:
import math
from typing import List, Tuple


# ══════════════════════════════════════════════════════════════════════
# Section 1: QCD Running Coupling vs Stribeck Curve
# ══════════════════════════════════════════════════════════════════════
#
# 1-loop QCD running coupling:
#   α_s(Q²) = 12π / ((33 - 2n_f) · ln(Q²/Λ²))
#
# Stribeck friction curve:
#   μ(v) = μ_k + (μ_s - μ_k) · exp(-(v/v_th)²)
#
# Both are smooth monotone transfer functions with a single threshold.
# We fit the Stribeck form to α_s and compute residuals.

LAMBDA_QCD = 0.2  # GeV


def alpha_s_1loop(Q: float, n_f: int = 3, Lambda: float = LAMBDA_QCD) -> float:
    """1-loop QCD running coupling.
    
    α_s(Q²) = 12π / ((33 - 2n_f) · ln(Q²/Λ²))
    
    Valid for Q >> Λ_QCD. Diverges at Q = Λ_QCD (Landau pole).
    """
    if Q <= Lambda:
        return float('inf')
    b0 = 33 - 2 * n_f
    return 12 * math.pi / (b0 * math.log(Q**2 / Lambda**2))


def stribeck(v: float, mu_s: float, mu_k: float, v_th: float) -> float:
    """Stribeck friction curve: μ(v) = μ_k + (μ_s - μ_k)·exp(-(v/v_th)²)."""
    return mu_k + (mu_s - mu_k) * math.exp(-(v / v_th) ** 2)


# Generate α_s data points for Q from 0.5 GeV to 1000 GeV
# Use n_f = 3 (below charm), n_f = 4 (charm-bottom), n_f = 5 (above bottom)
# Flavor thresholds: m_c ≈ 1.27 GeV, m_b ≈ 4.18 GeV, m_t ≈ 173 GeV

Q_values = [0.5 + i * 0.5 for i in range(2000)]  # 0.5 to 1000 GeV


def effective_n_f(Q: float) -> int:
    """Number of active quark flavors at energy Q."""
    if Q < 1.27:
        return 3
    elif Q < 4.18:
        return 4
    elif Q < 173.0:
        return 5
    else:
        return 6


alpha_data = []
Q_valid = []
for Q in Q_values:
    nf = effective_n_f(Q)
    a = alpha_s_1loop(Q, n_f=nf)
    if a < 2.0 and a > 0:  # exclude near-pole values
        alpha_data.append(a)
        Q_valid.append(Q)

print(f"Generated {len(alpha_data)} valid α_s data points")
print(f"Q range: {Q_valid[0]:.1f} to {Q_valid[-1]:.1f} GeV")
print(f"α_s range: {alpha_data[-1]:.4f} to {alpha_data[0]:.4f}")
print()

# Known experimental values for comparison
# PDG 2024: α_s(M_Z) = 0.1180 ± 0.0009, M_Z = 91.1876 GeV
alpha_s_MZ_exp = 0.1180
alpha_s_MZ_calc = alpha_s_1loop(91.1876, n_f=5)
print(f"α_s(M_Z) experimental (PDG 2024): {alpha_s_MZ_exp:.4f}")
print(f"α_s(M_Z) 1-loop calculation:      {alpha_s_MZ_calc:.4f}")
print(f"(1-loop overestimates — higher loops and matching corrections reduce it)")

In [ ]:
# ── Fit Stribeck form to α_s ─────────────────────────────────────────
#
# We fit: α_s(Q) ≈ μ_k + (μ_s - μ_k) · exp(-(Q/v_th)²)
#
# But α_s decays as 1/ln(Q²), not exponentially. So the fit will
# have systematic residuals. The QUESTION is: how large are they?
#
# Manual least-squares fit (no scipy needed):
# Fix μ_k = 0 (α_s → 0 at Q → ∞)
# Fit μ_s and v_th by grid search.

best_r2 = -float('inf')
best_params = (0, 0)

# Grid search over μ_s (max coupling) and v_th (threshold)
for mu_s_100 in range(20, 200, 2):  # μ_s from 0.2 to 2.0
    mu_s = mu_s_100 / 100.0
    for v_th_10 in range(2, 50, 1):  # v_th from 0.2 to 5.0 GeV
        v_th = v_th_10 / 10.0
        
        ss_res = 0.0
        ss_tot = 0.0
        mean_alpha = sum(alpha_data) / len(alpha_data)
        
        for i in range(len(Q_valid)):
            predicted = stribeck(Q_valid[i], mu_s, 0.0, v_th)
            ss_res += (alpha_data[i] - predicted) ** 2
            ss_tot += (alpha_data[i] - mean_alpha) ** 2
        
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0
        if r2 > best_r2:
            best_r2 = r2
            best_params = (mu_s, v_th)

mu_s_fit, v_th_fit = best_params
print("=== Stribeck Fit to QCD Running Coupling ===")
print()
print(f"Best-fit parameters:")
print(f"  μ_s (max coupling):  {mu_s_fit:.3f}")
print(f"  μ_k (min coupling):  0.000 (fixed)")
print(f"  v_th (threshold Q):  {v_th_fit:.1f} GeV")
print(f"  R² = {best_r2:.6f}")
print()

# Compute and display residuals at key energies
print(f"{'Q (GeV)':>10s}  {'α_s(Q)':>10s}  {'Stribeck':>10s}  {'residual':>10s}  {'% error':>10s}")
print("-" * 56)

key_Q = [1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 91.2, 200.0, 500.0]
residuals = []
for Q in key_Q:
    nf = effective_n_f(Q)
    a_qcd = alpha_s_1loop(Q, n_f=nf)
    a_strib = stribeck(Q, mu_s_fit, 0.0, v_th_fit)
    resid = a_qcd - a_strib
    pct = 100 * resid / a_qcd if a_qcd > 0 else 0
    residuals.append(resid)
    print(f"{Q:10.1f}  {a_qcd:10.4f}  {a_strib:10.4f}  {resid:10.4f}  {pct:10.1f}%")

print()
print("Key observation: the Stribeck exponential decay is FASTER than")
print("the 1/ln(Q²) decay of α_s. The Stribeck function underestimates")
print("α_s at intermediate Q and overestimates at high Q.")
print()
print("This is expected: α_s has a logarithmic tail (slow decay),")
print("while the Stribeck has an exponential tail (fast decay).")
print("They share the threshold structure but differ in the tail.")
print()
print("Both belong to the class of smooth monotone transfer functions")
print("with a single inflection point. The tails differ because the")
print("Stribeck is phenomenological while α_s derives from the RG flow.")

In [ ]:
# ── ASCII plot: α_s vs Stribeck fit ──────────────────────────────────

print("=== α_s(Q) vs Stribeck Fit (ASCII) ===")
print()
print("  α_s")
print("  ↑")

# Plot from Q = 0.5 to 20 GeV (where the interesting structure is)
plot_Q = [0.5 + i * 0.25 for i in range(80)]  # 0.5 to 20.5
max_alpha = 1.0
n_rows = 25
n_cols = 60

# Build grid
grid = [[' ' for _ in range(n_cols)] for _ in range(n_rows)]

for Q in plot_Q:
    col = int((Q - 0.5) / 20.0 * (n_cols - 1))
    if 0 <= col < n_cols:
        nf = effective_n_f(Q)
        a_qcd = alpha_s_1loop(Q, n_f=nf)
        a_strib = stribeck(Q, mu_s_fit, 0.0, v_th_fit)
        
        if 0 < a_qcd < max_alpha:
            row_qcd = n_rows - 1 - int(a_qcd / max_alpha * (n_rows - 1))
            if 0 <= row_qcd < n_rows:
                grid[row_qcd][col] = '●'
        
        if 0 < a_strib < max_alpha:
            row_strib = n_rows - 1 - int(a_strib / max_alpha * (n_rows - 1))
            if 0 <= row_strib < n_rows:
                if grid[row_strib][col] == '●':
                    grid[row_strib][col] = '⊕'  # overlap
                else:
                    grid[row_strib][col] = '○'

for i, row in enumerate(grid):
    alpha_label = f"{max_alpha * (n_rows - 1 - i) / (n_rows - 1):.2f}"
    print(f"  {alpha_label:>5s} │{''.join(row)}│")

print(f"        └{'─' * n_cols}┘")
print(f"         Q = 0.5{'':>{n_cols - 16}}Q = 20 GeV")
print()
print("  ● = α_s (QCD running coupling, 1-loop)")
print("  ○ = Stribeck best fit")
print("  ⊕ = overlap")
print()
print(f"  Λ_QCD = {LAMBDA_QCD} GeV")
print(f"  v_th (Stribeck fit) = {v_th_fit:.1f} GeV")
print(f"  R² = {best_r2:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Section 2: Clockwork Chain Lengths
# ══════════════════════════════════════════════════════════════════════
#
# The clockwork mechanism suppresses a UV coupling by q^N.
# Chain length: N = log(M_Planck / scale) / log(q)
#
# We compute N for multiple thresholds and gear ratios,
# then check ratios for integer or simple-fraction relationships.

# Physical scales (in GeV)
M_PLANCK = 1.22e19       # Planck mass
LAMBDA_QCD_GEV = 0.2     # QCD confinement scale
V_EW = 246.0             # Higgs VEV / electroweak scale

# a₀ in natural units: a₀/c² ≈ 1.33 × 10⁻²⁸ m⁻¹
# Converting to GeV via ℏc ≈ 0.197 GeV·fm = 1.97 × 10⁻¹⁶ GeV·m:
# a₀_natural = a₀/(c² · ℏc) ≈ 1.2e-10 / (9e16 · 1.97e-16) = 1.2e-10 / 1.77e1 ≈ 6.8e-12 m⁻¹
# Wait — let's be more careful.
# a₀ = 1.2e-10 m/s²
# In natural units where ℏ = c = 1:
# [acceleration] = [energy]² (since [length] = [energy]⁻¹, [time] = [energy]⁻¹)
# a₀ in eV²: a₀ = 1.2e-10 m/s² × (1/(ℏc²)) in appropriate units
# ℏc = 1.97e-7 eV·m, c = 3e8 m/s
# a₀ / (c²/ℏc) = a₀ · ℏc/c² = 1.2e-10 · 1.97e-7 / (9e16) = 2.63e-34 eV
# = 2.63e-43 GeV

# More precisely:
hbar_c = 1.9733e-16  # GeV·m
c_m_s = 2.9979e8     # m/s
a0_SI = 1.2e-10      # m/s²
a0_GeV_sq = a0_SI * hbar_c / (c_m_s ** 2)  # Convert to GeV² in natural units
# Actually: [a] = [length/time²] → in natural units [E²]
# a₀ (GeV²) = a₀ · ℏ/c³
hbar_SI = 1.0546e-34  # J·s
a0_natural_GeV2 = a0_SI * hbar_SI / (c_m_s ** 3)
# This gives units of GeV² (since ℏ/c³ converts m/s² to (energy)² in natural units)
# Actually let's just use the energy equivalent directly
# The relevant comparison is a₀ · R_typical ~ energy/mass ~ v²
# For clockwork, we need a₀ expressed as an energy scale.
# a₀ · ℏ/c = 1.2e-10 · 1.055e-34 / 3e8 = 4.2e-53 J = 2.6e-44 GeV
# So the "energy equivalent" of a₀ is ~ 10⁻⁴⁴ GeV

a0_energy_GeV = a0_SI * hbar_SI / c_m_s  # ℏa₀/c in GeV
a0_energy_GeV_precise = a0_energy_GeV / 1.602e-10  # convert J to GeV

# Let me just compute it step by step:
a0_energy_J = a0_SI * hbar_SI / c_m_s  # units: (m/s²)(J·s)/(m/s) = J
a0_energy_eV = a0_energy_J / 1.602e-19
a0_energy_GeV_final = a0_energy_eV / 1e9

print("=== Clockwork Chain Lengths ===")
print()
print("Physical scales:")
print(f"  M_Planck    = {M_PLANCK:.3e} GeV")
print(f"  v_EW (Higgs)= {V_EW:.1f} GeV")
print(f"  Λ_QCD       = {LAMBDA_QCD_GEV:.1f} GeV")
print(f"  a₀ (≡ ℏa₀/c)= {a0_energy_GeV_final:.3e} GeV")
print()

scales = {
    "v_EW": V_EW,
    "Λ_QCD": LAMBDA_QCD_GEV,
    "a₀": a0_energy_GeV_final,
}

gear_ratios = [2, 3, 5, 7]

print(f"{'Scale':>10s}  {'Energy (GeV)':>14s}", end="")
for q in gear_ratios:
    print(f"  {'N(q='+str(q)+')':>10s}", end="")
print()
print("-" * (26 + 12 * len(gear_ratios)))

chain_lengths = {}  # (scale_name, q) -> N

for name, energy in scales.items():
    log_ratio = math.log10(M_PLANCK / energy)
    print(f"{name:>10s}  {energy:14.3e}", end="")
    for q in gear_ratios:
        N = math.log(M_PLANCK / energy) / math.log(q)
        chain_lengths[(name, q)] = N
        print(f"  {N:10.2f}", end="")
    print()

In [ ]:
# ── Chain length ratios ──────────────────────────────────────────────

print()
print("=== Chain Length Ratios ===")
print()
print("If multiple thresholds arise from the same clockwork mechanism")
print("with the same gear ratio q, the ratios of chain lengths encode")
print("the hierarchy between forces.")
print()

scale_pairs = [
    ("Λ_QCD", "a₀"),
    ("v_EW", "Λ_QCD"),
    ("v_EW", "a₀"),
]

print(f"{'Ratio':>20s}", end="")
for q in gear_ratios:
    print(f"  {'q='+str(q):>10s}", end="")
print()
print("-" * (20 + 12 * len(gear_ratios)))

for s1, s2 in scale_pairs:
    label = f"N({s1})/N({s2})"
    print(f"{label:>20s}", end="")
    for q in gear_ratios:
        ratio = chain_lengths[(s1, q)] / chain_lengths[(s2, q)]
        print(f"  {ratio:10.4f}", end="")
    print()

print()
print("Key observation: the ratio N(s1)/N(s2) is INDEPENDENT of q.")
print("This is trivial: N = log(M_P/E) / log(q), so the ratio is")
print("log(M_P/E1) / log(M_P/E2), which cancels log(q).")
print()
print("The DIFFERENCE N(s2) - N(s1), however, depends on q:")
print()

print(f"{'Difference':>20s}", end="")
for q in gear_ratios:
    print(f"  {'q='+str(q):>10s}", end="")
print()
print("-" * (20 + 12 * len(gear_ratios)))

for s1, s2 in scale_pairs:
    label = f"N({s2})-N({s1})"
    print(f"{label:>20s}", end="")
    for q in gear_ratios:
        diff = chain_lengths[(s2, q)] - chain_lengths[(s1, q)]
        print(f"  {diff:10.2f}", end="")
    print()

print()
print("The ratio of DIFFERENCES:")
print("  [N(a₀) - N(Λ_QCD)] / [N(Λ_QCD) - N(v_EW)]")
print()
for q in gear_ratios:
    num = chain_lengths[("a₀", q)] - chain_lengths[("Λ_QCD", q)]
    den = chain_lengths[("Λ_QCD", q)] - chain_lengths[("v_EW", q)]
    ratio = num / den if den != 0 else float('inf')
    print(f"  q = {q}: ratio = {ratio:.4f}")

print()
print("This ratio is also q-independent (same cancellation).")
print("It equals log(Λ_QCD/a₀) / log(v_EW/Λ_QCD).")

ratio_fundamental = math.log(LAMBDA_QCD_GEV / a0_energy_GeV_final) / math.log(V_EW / LAMBDA_QCD_GEV)
print(f"\n  Fundamental ratio = {ratio_fundamental:.4f}")
print(f"  ≈ {ratio_fundamental:.1f}")
print()
print("Interpretation: the gap from electroweak to QCD is ~1/" + f"{ratio_fundamental:.1f}")
print("of the gap from QCD to the gravitational threshold a₀.")
print("The gravitational hierarchy is much larger than the strong-electroweak one.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Section 3: Quadratic Maximum Verification
# ══════════════════════════════════════════════════════════════════════
#
# Feigenbaum universality requires a quadratic maximum (z = 2).
# We verify this for:
#   (a) The MOND interpolating function μ(x) = x/(1+x)
#   (b) The QCD beta function β(α_s) = -(β₀/2π)α_s²

print("=== Quadratic Maximum Verification ===")
print()

# (a) MOND interpolating function
print("(a) MOND interpolating function: μ(x) = x/(1+x)")
print()
print("    Taylor expansion around x = 1 (the transition point a = a₀):")
print()
print("    μ(x) = x/(1+x)")
print("    μ(1) = 1/2")
print("    μ'(x) = 1/(1+x)²")
print("    μ'(1) = 1/4")
print("    μ''(x) = -2/(1+x)³")
print("    μ''(1) = -2/8 = -1/4  ≠ 0  ← QUADRATIC")
print("    μ'''(x) = 6/(1+x)⁴")
print("    μ'''(1) = 6/16 = 3/8")
print()

# Verify numerically
def mu_mond(x):
    return x / (1 + x)

h = 1e-6
x0 = 1.0
mu_pp_num = (mu_mond(x0 + h) - 2 * mu_mond(x0) + mu_mond(x0 - h)) / h**2
print(f"    Numerical μ''(1) = {mu_pp_num:.6f}")
print(f"    Analytical μ''(1) = {-2.0/8.0:.6f}")
print()
print("    → μ''(1) ≠ 0, so the composite map has a QUADRATIC maximum.")
print("    → Feigenbaum universality class: δ = 4.6692...")
print()

# Also check the alternative MOND function
print("    Alternative: μ(x) = x/√(1+x²)")
# μ'(x) = 1/(1+x²)^(3/2)
# μ''(x) = -3x/(1+x²)^(5/2)
# μ''(1) = -3/(2^(5/2)) = -3/(4√2) ≈ -0.530
mu_pp_alt = -3.0 / (2**2.5)
print(f"    μ''(1) = -3/(4√2) ≈ {mu_pp_alt:.4f}  ≠ 0  ← also QUADRATIC")
print()

In [ ]:
# (b) QCD beta function
print("(b) QCD beta function: β(α_s) = -(β₀/2π)α_s² (1-loop)")
print()

# β₀ = 11 - 2n_f/3
for nf in [3, 4, 5, 6]:
    beta0 = 11.0 - 2.0 * nf / 3.0
    print(f"    n_f = {nf}: β₀ = {beta0:.3f}")

print()
print("    β(α_s) = -(β₀/2π)·α_s²")
print("    dβ/dα_s = -(β₀/π)·α_s")
print("    d²β/dα_s² = -(β₀/π)  ≠ 0  ← QUADRATIC in α_s")
print()

# The beta function has a zero at α_s = 0 (UV fixed point).
# Near this zero, β ~ -α_s² (quadratic).
# The Feigenbaum universality class is determined by the order of
# the zero/maximum of the iterated map formed from the RG flow.

print("    Near the UV fixed point α_s = 0:")
print("    β(α_s) ~ -α_s²  (quadratic zero)")
print()
print("    If the RG flow near the deconfinement transition can be")
print("    mapped to a unimodal iterated map, the quadratic structure")
print("    of β guarantees the map's maximum is quadratic (z = 2).")
print("    → Same Feigenbaum universality class as gravity: δ = 4.6692")
print()
print("    ┌─────────────────────────────────────────────────────┐")
print("    │ Both the gravitational Stribeck curve (μ''(1) ≠ 0) │")
print("    │ and the QCD beta function (d²β/dα² ≠ 0) have       │")
print("    │ quadratic structure at their transition points.     │")
print("    │                                                     │")
print("    │ Feigenbaum universality class: δ = 4.6692...        │")
print("    │ This is a PREREQUISITE, not an inverse.             │")
print("    │ Quadratic maximum → period doubling with THIS δ.   │")
print("    └─────────────────────────────────────────────────────┘")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Section 4: Schwarzschild Mimetic Constraint Violation
# ══════════════════════════════════════════════════════════════════════
#
# Mimetic constraint: g^{μν} ∂_μφ ∂_νφ = 1
#
# For φ = φ(r) in Schwarzschild:
#   g^{rr} (dφ/dr)² = (1 - 2M/r)(dφ/dr)² = 1
#   → (dφ/dr)² = 1/(1 - 2M/r)
#
# At r = 2M (horizon): 1 - 2M/r = 0 → (dφ/dr)² diverges
# For r < 2M: 1 - 2M/r < 0 → (dφ/dr)² < 0 → impossible for real φ

print("=== Schwarzschild Mimetic Constraint Violation ===")
print()
print("Mimetic constraint: g^{μν} ∂_μφ ∂_νφ = 1")
print("For radial scalar field φ(r) in Schwarzschild geometry:")
print("  g^{rr} (φ')² = (1 - r_s/r)(φ')² = 1")
print("  → (φ')² = 1/(1 - r_s/r)")
print()

# Compute g^{rr} and required (φ')² as function of r/r_s
phi_prime_header = "(phi')²=1/g^{rr}"
print(f"{'r/r_s':>8s}  {'g^{rr}':>12s}  {phi_prime_header:>16s}  {'status':>20s}")
print("-" * 62)

r_ratios = [10.0, 5.0, 3.0, 2.0, 1.5, 1.1, 1.01, 1.001, 1.0, 0.999, 0.99,
            0.9, 0.5, 0.1, 0.01, 0.001]

for r_ratio in r_ratios:
    g_rr = 1.0 - 1.0 / r_ratio  # g^{rr} = 1 - r_s/r = 1 - 1/(r/r_s)
    
    if abs(g_rr) < 1e-10:
        phi_sq_str = "∞ (diverges)"
        status = "HORIZON — φ' diverges"
    elif g_rr < 0:
        phi_sq = 1.0 / g_rr
        phi_sq_str = f"{phi_sq:.4f} (< 0)"
        status = "IMPOSSIBLE (φ' imaginary)"
    else:
        phi_sq = 1.0 / g_rr
        phi_sq_str = f"{phi_sq:.4f}"
        status = "satisfiable"
    
    print(f"{r_ratio:8.3f}  {g_rr:12.6f}  {phi_sq_str:>16s}  {status:>20s}")

print()
print("Three regimes:")
print("  r > r_s: constraint satisfiable (g^{rr} > 0, (φ')² > 0 and finite)")
print("  r = r_s: constraint requires (φ')² → ∞ (coordinate singularity)")
print("  r < r_s: constraint requires (φ')² < 0 — IMPOSSIBLE for real scalar")
print()
print("In Lagrangian relaxation language:")
print("  r > r_s: constraint set is nonempty → dual variable (λ) finite")
print("  r ≤ r_s: constraint set is EMPTY → Slater's condition fails")
print("           → dual variable DIVERGES")
print("           → the 'cost' of enforcing the geometric constraint is infinite")
print()
print("The singularity is not a point of infinite density.")
print("It is a region where the geometric constraint is infeasible")
print("and the shadow price of enforcement diverges.")

In [ ]:
# ── ASCII plot: constraint violation vs r/r_s ────────────────────────

print("=== Constraint Violation vs r/r_s (ASCII) ===")
print()
print("  (φ')² = 1/g^{rr} = 1/(1 - r_s/r)")
print()

n_rows = 20
n_cols = 60

# Plot r/r_s from 1.01 to 10 (outside horizon only — inside is unphysical)
r_min, r_max = 1.01, 10.0
# (φ')² ranges from ~1.01 at r/r_s=100 to ~100 near horizon
# Use log scale for (φ')²
log_phi_min = 0.0  # log10(1) = 0
log_phi_max = 2.5  # log10(~300)

grid = [[' ' for _ in range(n_cols)] for _ in range(n_rows)]

for col in range(n_cols):
    r_ratio = r_min + (r_max - r_min) * col / (n_cols - 1)
    g_rr = 1.0 - 1.0 / r_ratio
    if g_rr > 0:
        phi_sq = 1.0 / g_rr
        log_phi = math.log10(phi_sq)
        row = n_rows - 1 - int((log_phi - log_phi_min) / (log_phi_max - log_phi_min) * (n_rows - 1))
        if 0 <= row < n_rows:
            grid[row][col] = '●'

# Add horizon marker
for row in range(n_rows):
    grid[row][0] = '║'  # horizon at leftmost column

print("  log₁₀(φ')²")
print("  ↑")
for i, row in enumerate(grid):
    val = log_phi_max - (log_phi_max - log_phi_min) * i / (n_rows - 1)
    print(f"  {val:5.1f} │{''.join(row)}│")

print(f"        └{'─' * n_cols}┘→ r/r_s")
print(f"        r_s{' ' * (n_cols - 8)}10·r_s")
print()
print("  ║ = event horizon (r = r_s)")
print("  ● = required (φ')² to satisfy mimetic constraint")
print()
print("  At the horizon: (φ')² → ∞  (constraint unsatisfiable)")
print("  Far from BH:    (φ')² → 1   (constraint trivially satisfied)")

In [ ]:
# ── Kretschner scalar for comparison ────────────────────────────────
#
# The Kretschner scalar K = R_{μνρσ}R^{μνρσ} = 48M²/r⁶
# quantifies the actual curvature singularity at r → 0.
# Compare its divergence rate to the constraint violation.

print("=== Divergence Comparison: Kretschner vs Constraint Violation ===")
print()
print("At the singularity (r → 0):")
print("  Kretschner scalar: K = 48M²/r⁶  →  diverges as r⁻⁶")
print("  Constraint violation: (φ')² = 1/(1-r_s/r)  →  for r < r_s,")
print("    (φ')² = 1/(1-r_s/r) → as r→0, 1-r_s/r → -∞, so (φ')² → 0⁻")
print("    (approaches zero from below — imaginary φ')")
print()
print("  The constraint CHANGES CHARACTER at the horizon:")
print("  - Outside: real φ' required, increases toward horizon")
print("  - At horizon: φ' → ∞ (diverges)")
print("  - Inside: no real φ' can satisfy the constraint")
print()

phi_header = "(phi')² needed"
print(f"{'r/r_s':>8s}  {'g^rr':>10s}  {phi_header:>14s}  {'K/K0':>12s}  {'dual var':>10s}")
print("-" * 60)

K0 = 48.0  # K at r = r_s (in units where M=1, r_s=2M=2 → K₀ = 48/64 = 0.75)

for r_ratio in [10.0, 5.0, 2.0, 1.5, 1.1, 1.01, 1.001]:
    g_rr = 1.0 - 1.0 / r_ratio
    phi_sq = 1.0 / g_rr
    K_ratio = 1.0 / r_ratio**6  # K ∝ 1/r⁶
    
    # Dual variable: penalty for constraint violation
    # In the relaxed problem, λ grows when constraint is tight
    # λ ∝ (φ')² - 1 = 1/g^{rr} - 1 = (1 - g^{rr})/g^{rr} = (r_s/r)/(1-r_s/r)
    lam = (1.0 / r_ratio) / g_rr if g_rr > 0 else float('inf')
    
    print(f"{r_ratio:8.3f}  {g_rr:10.6f}  {phi_sq:14.4f}  {K_ratio:12.2e}  {lam:10.4f}")

print()
print("Near the horizon (r → r_s):")
print("  - The dual variable λ diverges as 1/(r - r_s)")
print("  - The curvature K diverges (at the singularity, not the horizon)")
print("  - The constraint infeasibility (inside horizon) is a SEPARATE")
print("    issue from the curvature singularity (at r = 0)")
print()
print("The mimetic framework identifies TWO pathologies:")
print("  1. Constraint divergence at r = r_s (dual variable → ∞)")
print("  2. Constraint infeasibility for r < r_s (no real solution)")
print("Both are standard results in constrained optimization (Boyd & Vandenberghe, 2004).")

## Summary

1. **QCD vs Stribeck:** Both are smooth monotone transfer functions with a single threshold. The Stribeck exponential fits the QCD running coupling well near the threshold but underestimates the logarithmic tail at high Q. They share structural class but differ in asymptotic behavior.

2. **Clockwork chain lengths:** The ratios N(scale₁)/N(scale₂) are q-independent (trivially). The fundamental ratio log(Λ_QCD/a₀)/log(v_EW/Λ_QCD) characterizes the relative hierarchy gaps between force scales.

3. **Quadratic maximum:** Both the MOND interpolating function μ(x) = x/(1+x) and the QCD beta function β(α_s) = -(β₀/2π)α_s² have quadratic structure at their transition points (μ''(1) ≠ 0, d²β/dα² ≠ 0). Both belong to the Feigenbaum universality class δ = 4.6692. Quadratic maximum is the prerequisite; period doubling is the consequence.

4. **Schwarzschild constraint:** The mimetic constraint g^{μν}∂_μφ∂_νφ = 1 requires (φ')² = 1/(1-r_s/r). This diverges at the horizon and becomes unphysical (negative) inside. In optimization language: the constraint set is empty for r ≤ r_s, Slater's condition fails, and the dual variable diverges.

---

*Standard library only. No external dependencies. CC0.*